### Imports

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from catboost import Pool, CatBoostClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_curve, RocCurveDisplay, precision_recall_curve, PrecisionRecallDisplay
from imblearn.under_sampling import RandomUnderSampler
from IPython.display import clear_output
import time
from datetime import timedelta
from tqdm import tqdm

sns.set(context='notebook', style='whitegrid')
plt.rcParams['text.usetex'] = True

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

### Hyperparamater tuner

In [ ]:
class CatTune():
    '''
    Tunes the paramaters of a catboost model. 
    Also tunes the rate of undersampling and the respective class weights.
    '''
    def __init__(self, X, y, cat_dict, cv):
        # Set attributes
        self.cat_dict = cat_dict
        self.cv = cv
        self.cat_keys = list(cat_dict.keys())
        self.sample_names = ['sample', 'weight0', 'weight1']
        self.X = X
        self.y = y
        # Create list of all possible paramater dictionaries
        self.get_search_space()
   
    def get_search_space(self):
        # Get a list of all posible paramater combinations
        search_list = list(itertools.product(*self.cat_dict.values()))
        # Get number of paramater comninations to be trialled 
        self.n_fits = len(search_list)
        # Create list of dictionaries to be iterated over
        self.search_space = [{self.cat_keys[i]:j for i, j in enumerate(k)} for k in search_list]
        
    def split_params(self, p):
        # Splits a paramater dictionary into sampling and catboost paramaters 
        sample_dict = {self.cat_keys[i]:j for i, j in enumerate(p.values()) if self.cat_keys[i] in self.sample_names}
        cat_dict = {self.cat_keys[i]:j for i, j in enumerate(p.values()) if self.cat_keys[i] not in self.sample_names}
        return sample_dict, cat_dict
    
    def make_training_data(self, X_train, y_train, sample, weight0, weight1):
        # Undersample tranining data
        X_tr, y_tr = RandomUnderSampler(sampling_strategy=sample, random_state=0).fit_resample(X_train, y_train)
        # Create sample weights
        weight=compute_sample_weight({0:weight0, 1:weight1}, y_tr)
        # Create catboost pool object
        train_data = Pool(
            data=X_tr,
            label=y_tr,
            weight=weight
        )
        return train_data
    
    def tune(self):
        # Initialise best paramaters and scoring 
        self.best_roc = 0
        self.best_iter = 0
        self.best_params = None
        self.best_classifier = None
        # Counter for total time taken by the tuning
        self.tot_time = 0
        # Initialise empty array to store ROC AUC score
        self.roc_array = np.empty(self.n_fits)
        # Start time of the tuning
        st = time.time()
        print(f'Starting tuning: n_fits = {self.n_fits}, cv = {self.cv}, total fits = {self.n_fits*self.cv}')
        # Iterating through every paramater combination
        for i in range(self.n_fits):
            # Start time for the iteration
            s = time.time()
            # Splitting the paramater dictionaty into sample and catboost paramaters
            sample_dict, cat_dict = self.split_params(self.search_space[i])
            current_roc = 0
            # Cross validate for self.cv folds
            for j, (train_idx, test_idx) in enumerate(KFold(n_splits=self.cv).split(self.X)):
                # Create the training data for the sample params
                train_data = self.make_training_data(self.X[train_idx], self.y[train_idx], **sample_dict)
                # Initialise CatBoost classifier
                cat = CatBoostClassifier(**cat_dict)
                # Fit the classifier on the model
                cat.fit(train_data, verbose=0)
                # Predict the class of the training fold
                y_pred = cat.predict_proba(self.X[test_idx])
                # Add the roc score of the classifier
                current_roc += roc_auc_score(self.y[test_idx], y_pred[:, 1])
            # Get the average score over the cv folds
            current_roc /= self.cv
            # Append current roc to array
            self.roc_array[i] = current_roc
            # Update best parameters
            if self.roc_array[i] > self.best_roc:
                self.best_roc = self.roc_array[i]
                self.best_iter = i
                self.best_params = self.search_space[i]
                self.best_classifier = cat
            # End time
            e = time.time()
            # Calculate average time remaining
            self.tot_time += (e - s)
            current_avg = self.tot_time/(i+1)
            remaining = (self.n_fits - i+1) * current_avg
            clear_output(wait=True)
            # Update display
            print('█' * int(100*((i)/self.n_fits)) + f' {100*(i/self.n_fits):.2f}%')
            print(f'{i+1}/{self.n_fits} fits, Current ROC AUC score: {self.roc_array[i]:.3f}, Best ROC AUC score: {self.best_roc:.3f}')
            print(f'Average time per iter: ', str(timedelta(seconds=current_avg))[:7] ,', Time remaining:',str(timedelta(seconds=remaining)), 'Time elapsed:', str(timedelta(seconds=e-st))[:7])
        # Split the paramaters into sample and catboost
        self.best_sample_params, self.best_cat_params = self.split_params(self.best_params)
            
        print(f'Finished tuning! Best roc : {self.best_roc}') 

### Feature engineering function

In [ ]:
def create_features(df):
    '''
    Calculates various statistics along columns 
    PARAMS:
    df : pandas.core.frame.DataFrame
    RETURNS:
    df : pandas.core.frame.DataFrame
    '''
    features = [feat for feat in df.columns if 'V' in feat]
    df['V_sum'] = df[features].sum(axis = 1)
    df['V_min'] = df[features].min(axis = 1)
    df['V_max'] = df[features].max(axis = 1)
    df['V_avg'] = df[features].mean(axis = 1)
    df['V_std'] = df[features].std(axis = 1)
    df['V_pos'] = df[features].gt(0).sum(axis = 1)
    df['V_neg'] = df[features].lt(0).sum(axis = 1)
    df['V_range'] = abs(df['V_min'] - df['V_max'])

    return df

### Loading data and feature engineering

In [ ]:
train_data = pd.read_csv('/kaggle/input/playground-series-s3e4/train.csv')
test_data = pd.read_csv('/kaggle/input/playground-series-s3e4/test.csv')

# Create features on the data
train_data = create_features(train_data)
test_data = create_features(test_data)

### Get features and scale data

In [ ]:
# Get features and labels
X = train_data[train_data.columns[train_data.columns != 'Class']].values
y = train_data.Class.values

# Scale data
X = StandardScaler().fit_transform(X)

### Hyperparamater tuning

In [ ]:
# Create paramater grid 
grid = {
    # Catboost params
    'eval_metric' : ['AUC'],
    'learning_rate':[0.01],
    'l2_leaf_reg':[5],
    'random_state':[1234],
    'n_estimators':np.linspace(600, 700, 3, dtype=np.int64),
    'max_depth':  [6],
    # Sample params
    'sample':np.linspace(0.01, 1, 2),
    'weight0':np.linspace(0.1, 1, 2),
    'weight1':np.linspace(0.1, 1, 2)
}

# Initialise the  CatTune class
cattune = CatTune(X, y, grid, cv=4)

# Tune
cattune.tune()

### Previous results from a longer run

In [ ]:
best_params = {
    'eval_metric': 'AUC',
    'learning_rate': 0.01,
    'l2_leaf_reg': 5,
    'random_state': 1234,
    'n_estimators': 655,
    'max_depth': 6,
    'sample': 0.12105263157894737,
    'weight0': 0.6000000000000001,
    'weight1': 0.5
}

In [ ]:
def random_jump(params):
    '''
    Jump around the hyperparamater space
    '''
    new_params = params.copy()
    new_params['learning_rate'] = params['learning_rate'] + np.abs(np.random.normal(0, 0.001))
    new_params['l2_leaf_reg'] = params['l2_leaf_reg'] + np.round(np.random.normal(0, 1))
    new_params['n_estimators'] = params['n_estimators'] + np.round(np.random.normal(0, 5))
    new_params['max_depth'] = params['max_depth'] + np.round(np.random.normal(0, 1))
    new_params['sample'] = params['sample'] + np.random.normal(0, 0.01)
    new_params['weight0'] = params['weight0'] + np.random.normal(0, 0.1)
    new_params['weight1'] = params['weight1'] + np.random.normal(0, 0.1)
    return new_params
    

def random_sample(params, iterations):
    '''
    Average over random jumps in hyperparamater space
    '''
    # Get test data
    X_test = StandardScaler().fit_transform(test_data.values)
    # Array for predictions
    predictions = np.empty((X_test.shape[0], 2))
    for i in tqdm(range(iterations)):
        # Make a random jump in hyperparamater space
        sample_dict, cat_dict = cattune.split_params(random_jump(best_params))
        # Train a classifier
        cat = CatBoostClassifier(**cat_dict)
        train_pool = cattune.make_training_data(X, y, **sample_dict)
        cat.fit(train_pool, verbose=0)
        # Add predictctions
        predictions += cat.predict_proba(X_test)
    # Return the average prediction
    return predictions / iterations

In [ ]:
y_pred = random_sample(best_params, 500)

In [ ]:
# Save submissions
submission = pd.DataFrame()

submission['id'] = test_data.id
submission['Class'] = y_pred[:, 1]

submission.to_csv('submission.csv', index=False)